In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Acting: action values, exploration, Q-learning and SARSA

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position: Part A (value-based methods), session 4**
    - values and the Bellman equation, dynamic programming, learning from samples, then **acting (this session)**, then a network instead of a table
    - learning the policy directly is a separate part that follows
    - **the limitation we start from:** Monte Carlo and TD(0) only *evaluate* the policy that plays; they do not make it better
    - this session learns to *act*: the values of actions, how to try them, and which policy an update learns about
- **This session's goal**
    - learn a good policy on Frozen Lake from experience alone
- **By the end you can**
    - say why state values are not enough to act without a model, and what action values add
    - run an $\epsilon$-greedy policy ($\epsilon$ is defined in the exploration section) and say why it explores
    - compute a Q-learning and a SARSA update by hand
    - name the *behavior policy* and the *target policy* in an update, and say which methods are on-policy and which off-policy
    - say why SARSA and Q-learning choose different paths on a cliff-edge grid (introduced in the SARSA section)


# Evaluating a policy is not yet acting

**Why this section?** The last step produced $\statevalfun_\pi$ for a policy $\pi$ that was playing. Two things stand between those values and a good policy.

- **Problem 1: acting on state values needs the model**
    - to pick the best action in $\state$ with $\statevalfun$, try each action and weight the possible next states by their probabilities under $\transp$
    - without the model that one-step lookahead cannot be computed
- **Problem 2: the experience depends on the policy that plays**
    - an action that is never tried never gets a value
    - whatever the agent does decides which states and rewards it sees
- **The plan**
    - learn the value of *actions* instead of states: that removes problem 1
    - choose actions so that every action keeps being tried: that is problem 2
    - the question left open last step ("must the episodes come from the policy we evaluate?") is answered in the on-policy / off-policy section


## Action values

**Why this section?** Storing a value for each state-*action* pair removes the lookahead: the best action is read off directly.

Define an *action value function* $\actvalfun_\pi(\state, \act)$
$$
\actvalfun_{\pi}: \States \times \Actions \to \Reals
$$

to map a state/action pair $(\state, \act)$ to the expected return of taking action $\act$ in state $\state$ and following $\pi$ afterwards.

- with $\actvalfun$, the best action is simply $\argmax{\act} \actvalfun(\state, \act)$: no model is needed for that step
- acting greedily on $\actvalfun_\pi$ gives a policy at least as good as $\pi$: the *policy improvement* step of policy iteration, now without the lookahead
- $\actvalfun_*$ is the action value when the policy is optimal, and $\statevalfun_*(\state) = \max{\act}\,\actvalfun_*(\state, \act)$


The Bellman equations for the action value
- the expectation form (for a given policy $\pi$), the one SARSA will sample
$$
\actvalfun_\pi(\state, \act) = \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc \sum_{\act'} \pi(\act' \mid \state')\,\actvalfun_\pi(\state', \act')\bigr)
$$
- the optimality form, which value iteration for $\actvalfun$ turns into an update (with the model), and which Q-learning will sample
$$
\actvalfun_*(\state, \act) = \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc \max{\act'} \actvalfun_*(\state', \act')\bigr)
$$
- from *sampled* transitions (state, action, reward, next state) the same updates can be made without $\transp$: that is what the rest of the session does


## Greed is not (always) good

**Why this section?** Action values fix problem 1 but not problem 2: a value exists only for actions that were tried. How should the agent choose its actions while it learns?

Before learning
- all values are $0$, so every action ties for best
- a *greedy* agent (always the action with the best current value) breaks the tie the same way every time
    - say, always the first action: 'left', which just bumps into the wall
- the action chosen in the first episode may not be the optimal one, and a deterministic policy locks us into it


This is the *exploration / exploitation* trade-off
- choosing the action currently believed to be best is **exploitation** (the *greedy* choice)
- choosing a different action is **exploration** (a *non-greedy* choice)

In model-free methods we may be forced to take actions that are not best by the current values, in order to sample the environment better.

An **$\epsilon$-greedy** policy makes
- the greedy choice with probability $(1-\epsilon)$
- a random action with probability $\epsilon$ (which may also be the greedy one)
- usually $\epsilon$ starts large and decays as the values improve


## Q-learning: learning action values from samples

**Why Q-learning?** There is no new idea here: it applies the sampling idea of the last step to the equation that finds the best policy, with action values so that no model is needed to act.

The Bellman equation has two forms (value slides), and dynamic programming used both with the model. Sampling replaces the sum over all outcomes by *one* observed outcome. Where the methods sit:

| | **evaluate a given policy** (Bellman expectation form) | **find the best policy** (Bellman optimality form, with a max) |
|:--|:--|:--|
| **with the model** (sum over all outcomes) | policy evaluation | value iteration |
| **from samples** (one outcome per update) | TD(0) for $\statevalfun_\pi$; SARSA for $\actvalfun_\pi$ | **Q-learning** for $\actvalfun_*$ |

- last step's TD(0) was the sampled version of the left column, for $\statevalfun$
- Q-learning is the sampled version of the right column, for $\actvalfun$
- the purpose of the comparison: Q-learning is TD's sampling idea applied to the optimality equation; SARSA, later, fills the remaining cell

The optimal action value satisfies the Bellman optimality equation
$$
\actvalfun_*(\state, \act) = \E\left[ \rewseq_{\tt+1} + \disc \max{\act'} \actvalfun_*(\stateseq_{\tt+1}, \act') \mid \stateseq_\tt = \state, \actseq_\tt = \act \right]
$$

Q-learning replaces the expectation by *one sample* and moves the estimate toward it
$$
\actvalfun(\stateseq_\tt, \actseq_\tt) \leftarrow \actvalfun(\stateseq_\tt, \actseq_\tt) + \alpha \left[ \rewseq_{\tt+1} + \disc \max{\act'} \actvalfun(\stateseq_{\tt+1}, \act') - \actvalfun(\stateseq_\tt, \actseq_\tt) \right]
$$
- the bracket is a TD error, now for action values: the **target** (recall: the number the update moves the estimate toward) minus the current estimate
    - here the target is one sampled reward plus the discounted best value of the next state, a TD target, so it has the bias and variance profile of TD (biased while the table is wrong, low variance)
- the target uses the best action in the next state, whichever action the agent actually takes next
    - with $\epsilon$-greedy the agent sometimes takes another (exploratory) action, so the action the target assumes can differ from the action taken: that gap is what *off-policy* means, named two sections on
- the policy is read off the table: $\pi(\state) = \argmax{\act} \actvalfun(\state, \act)$, the *greedy* policy of the current $\actvalfun$


A Q-learning update by hand, on (non-slippery) Frozen Lake
- all values start at $0$; $\alpha = 0.9$, $\disc = 0.9$
- the agent moves from square $14$ into **G** and earns $+1$
    - target $= 1 + 0.9 \times 0 = 1$ (the next state is terminal)
    - $\actvalfun(14, \text{right}) = 0 + 0.9\,(1 - 0) = 0.9$
- in a later episode it moves from square $13$ to square $14$ (reward $0$)
    - target $= 0 + 0.9 \times \max{\act'} \actvalfun(14, \act') = 0.9 \times 0.9 = 0.81$
    - $\actvalfun(13, \text{right}) = 0 + 0.9\,(0.81 - 0) \approx 0.73$
- information moves back one square each time a pair is revisited


**Pseudo code for Q-learning**

- initialize $\actvalfun(\state, \act)$ arbitrarily for all states and actions (often $0$); set the learning rate $\alpha$, the discount $\disc$ and the exploration rate $\epsilon$
- for each episode
    - initialize the state $\state$
    - repeat until $\state$ is terminal
        - with probability $\epsilon$ choose a random action $\act$ (exploration); otherwise choose $\act = \argmax{\act'} \actvalfun(\state, \act')$ (exploitation)
        - take action $\act$; observe the reward $\rew$ and the next state $\state'$
        - $\actvalfun(\state, \act) \leftarrow \actvalfun(\state, \act) + \alpha \bigl[ \rew + \disc \max{\act'} \actvalfun(\state', \act') - \actvalfun(\state, \act) \bigr]$ (if $\state'$ is terminal, the max term is $0$)
        - $\state \leftarrow \state'$

- the exploration rate $\epsilon$ is shown as a constant; in practice it usually decays over the episodes
- the table $Q$ is an estimate of $\actvalfun_*$; the algorithm never uses $\transp$ or any reward other than the one received
- the action $a$ comes from the $\epsilon$-greedy rule; the *update* assumes the next choices are greedy (the $\max{}$)


### Q-learning in action: Frozen Lake

**Why this section?** We have the algorithm. Now we watch it learn on the same Frozen Lake as the last sessions, and compare with the values value iteration computed.

Here is the value table before Q-learning begins (the policy and values are derived from the Q-table).

<br>
<table>
    <center><strong>Value function (uninitialized) as a table</strong></center>
<tr>
    <td><img src="images/Frozen_Lake_env.png"></td>
    <td><img src="images/qlearning_policy_evolution_0.png"></td>
</tr>
</table>


Here is an evolution in the early episodes of Q-learning training.

<br>
<table>
    <center><strong>Value function: Q-learning early episodes</strong></center>
<tr>
    <td><center><strong>Early episode </strong><br><br>It takes many episodes for the reward to diffuse from G<br>We need to experience an episode that reaches the neighbor of G</center></td>
    <td><center><strong>Slightly later episode</strong></center></td>
</tr>

<tr>
    <td><img src="images/qlearning_policy_evolution_70.png"></td>
    <td><img src="images/qlearning_policy_evolution_145.png"></td>
</tr>
</table>

Hotter (yellower) colors represent higher values; the color scale differs between the pictures.

- left: the reward information has reached the neighbor of the goal (square $14$), and the arrow there points to **G**
- right: the information has spread farther, and the policy follows it


**After training, and the success rate**

<table>
<tr>
    <td><center><strong>The table at the end of training</strong><br>(episode $19{,}999$)</center></td>
    <td><center><strong>Average reward per episode</strong><br>(success rate, smoothed over 100 episodes)</center></td>
</tr>
<tr>
    <td><img src="images/qlearning_policy_evolution_final.png"></td>
    <td><img src="images/Frozen_Lake_Qlearning_return_vs_episode.png"></td>
</tr>
</table>

- the arrows are the greedy actions; the start's value is $0.59$, the optimal value from the dynamic programming session
- not fully converged, compared with the optimal values of the last sessions: square $1$ reads $0.53$ (optimum $0.66$) and square $10$ reads $0.73$ (optimum $0.90$), so their arrows are not the best action; square $6$ reads $0.59$ (optimum $0.81$); squares $2$ and $3$ were rarely visited, so they still read $0$
- the success rate counts *training* episodes, exploratory moves included; it is not the discounted return ($0.9^5 = 0.59$): each episode earns $1$ or $0$
- it rises to about $1$ within roughly the first $1000$ of the $20{,}000$ episodes, as $\epsilon$ decays and the table improves

**Compared with dynamic programming**
- value iteration: a handful of sweeps over all squares, but it needs the model
- Q-learning: thousands of episodes of experience, but no model
- both end with the same value at the start square, $0.59$


**Aside (for your own review): the Frozen Lake run behind these pictures, and the animation**

- the same grid as before, squares numbered $0$ to $15$; reward $+1$ on entering **G**, episodes end at **G** or a hole
- **non-slippery**, discount $\disc = 0.9$, learning rate $\alpha = 0.9$ (a large step size, which works here because nothing is random), each episode cut off after $100$ steps
- $\epsilon$-greedy, with $\epsilon = 0.01 + 0.99\,e^{-0.005 \cdot \text{episode}}$: about $0.4$ after $200$ episodes, about $0.02$ after $1000$
- $20{,}000$ episodes in total; the table starts at $0$
- picture legend: pale blue is the start, grey is frozen, black is a hole, green is the goal; each square shows its best action value and the arrow is the greedy action; the goal shows $0.00$ because it is terminal; the actions are numbered left, down, right, up
- the animation plays once (about 50 seconds) and does not replay without reloading the page: the table after each episode, from $0$ to $19{,}999$

<img src="images/q_table_evolution_refined.gif" width=45%>


# Two policies in one update: on-policy and off-policy

**Why this section?** Last step we kept one question in mind: do the episodes have to come from the policy whose values we learn? Q-learning already says no. Naming the two policies makes the answer precise, and lets us compare it with its smallest variant, SARSA.

$$
\actvalfun(\state, \act) \leftarrow \actvalfun(\state, \act) + \alpha \Bigl[ \rew + \disc \, \underbrace{\max{\act'} \actvalfun(\state', \act')}_{\text{best next action}} - \actvalfun(\state, \act) \Bigr]
$$

Every update involves two policies
- the **behavior policy** picks the actions that are *actually taken*, so it generates the data
    - in Q-learning: $\epsilon$-greedy; it chose the action $\act$ whose value is updated
- the **target policy** is the policy whose value the update *learns about*
    - in Q-learning: the *greedy* policy: the $\max{\act'}$ always evaluates the best next action, whether or not the agent takes it
- the next action the agent really takes in $\state'$ (again $\epsilon$-greedy, possibly a random one) is *never used* in the update


When the behavior and target policies are identical
- the method is **on-policy**

If the behavior and target policies can differ
- the method is **off-policy**

**The answer to last step's question is no**: Q-learning is **off-policy**
- the data come from one policy ($\epsilon$-greedy) and the update learns about another (greedy)
- this holds even for the plain table: nothing is stored or reused
- Monte Carlo and TD(0) as presented last step are on-policy: they evaluate the policy that played

Off-policy is about *which policy is learned*, not about model-free versus model-based.


**Aside (for your own review): why Q-learning needs no correction for the difference between the policies**

- $\actvalfun(\state, \act)$ conditions on the *first* action, which is fixed by the data
- the $\max{}$ then handles every later action by *definition*, not by sampling
- so no importance weights (re-weighting by how likely each policy was to take the action) are needed in the one-step case
- methods that sample the target policy's actions, and that reuse data from an older policy, do need such a ratio: it appears again in the policy gradient part


# SARSA: the on-policy counterpart

**Why this section?** With the two policies named, the smallest change to Q-learning is to learn about the policy that is actually acting. That is SARSA.

SARSA is a small change to the Q-learning update
- Q-learning target: $\rewseq_{\tt+1} + \disc \max{\act'} \actvalfun(\stateseq_{\tt+1}, \act')$
- SARSA target: $\rewseq_{\tt+1} + \disc \, \actvalfun(\stateseq_{\tt+1}, \actseq_{\tt+1})$
    - $\actseq_{\tt+1}$ is the action the behavior policy *actually chooses* next
$$
\actvalfun(\stateseq_\tt, \actseq_\tt) \leftarrow \actvalfun(\stateseq_\tt, \actseq_\tt) + \alpha \bigl[ \rewseq_{\tt+1} + \disc \, \actvalfun(\stateseq_{\tt+1}, \actseq_{\tt+1}) - \actvalfun(\stateseq_\tt, \actseq_\tt) \bigr]
$$

SARSA is the *sampled* version of the Bellman equation for $\actvalfun_\pi$ (the expectation form, in the action-values section), just as TD(0) was for $\statevalfun_\pi$

The name comes from the quintuple the update uses: $\stateseq_\tt, \actseq_\tt, \rewseq_{\tt+1}, \stateseq_{\tt+1}, \actseq_{\tt+1}$
- the exploratory choices are *inside* the values SARSA learns
- so it learns the value of the policy it follows: *on-policy*

        Choose a from s with the epsilon-greedy policy
        Repeat for each step:
            Take a, observe r and s'
            Choose a' from s' with the epsilon-greedy policy
            Q(s, a) = Q(s, a) + alpha * [r + gamma * Q(s', a') - Q(s, a)]     (Q = 0 at terminal states)
            s = s'; a = a'


## Grid-world: SARSA vs Q-learning on the cliff

**Why this section?** Does the difference between the two methods matter in practice? We need a world where exploring is costly: a cliff. Its description comes first, then one transition on its edge, then the results.

We need to navigate in a $(4 \times 12)$ grid
- from start cell $S$
- to goal cell $G$
- avoiding the cliff cells: stepping on one costs $-100$ (instead of $-1$), sends the agent back to $S$, and the episode continues

Both learners use tables (no neural network).

There is a negative reward ($-1$) for each time step so the reward is maximized by getting to the goal quickly.

There are four moves (up, right, down, left); the episode ends at $G$. Both learners explore $\epsilon$-greedily.


One transition, two targets (illustrative numbers)
- the agent walks along the edge of the cliff and, after a step with reward $\rewseq = -1$, lands in $\state'$, also on the edge
- the values there: $\actvalfun(\state', \text{right}) = -8$ (keep walking) and $\actvalfun(\state', \text{down}) = -100$ (into the cliff); $\disc = 1$
- the numbers are chosen for illustration; in the grid-world a cliff step also sends the agent back to the start

What are the Q-learning and the SARSA targets if the behavior policy's next action is an exploratory step into the cliff?


| the next action chosen by the behavior policy | Q-learning target | SARSA target |
|:--|:-:|:-:|
| right (the greedy action) | $-1 + (-8) = -9$ | $-1 + (-8) = -9$ |
| down, into the cliff (an exploratory step) | $-1 + (-8) = -9$ | $-1 + (-100) = -101$ |

Q-learning ignores the slip; SARSA includes it, so the values along the edge drop.

| Method | Behavior policy | Target policy | |
|:--|:--|:--|:--|
| Q-learning | $\epsilon$-greedy | greedy | off-policy |
| SARSA | $\epsilon$-greedy | the same $\epsilon$-greedy policy | on-policy |


Q-learning and SARSA therefore learn *different things*
- Q-learning learns the value of the **greedy** policy, even though it acts $\epsilon$-greedily
- SARSA learns the value of the **$\epsilon$-greedy** policy it actually follows, exploratory mistakes included

When exploration is harmless, both end up with similar values.
When exploration is *costly* (a cliff), the values differ, and so do the paths.

With a *fixed* $\epsilon$, SARSA's table is the value of the $\epsilon$-greedy policy, not $\actvalfun_*$ (it reaches $\actvalfun_*$ only if $\epsilon$ also decays: see the aside)

The cliff contrast below is for a *fixed* $\epsilon$: exploration stays on.


<img src="images/cliff_sarsa_vs_qlearning.png" width=100%>

- left: Q-learning's greedy path is the 13-step edge route; SARSA's is usually a 17-step detour away from the edge (details in the aside)
- right: while exploring ($\epsilon = 0.1$), Q-learning earns *less* reward per episode, because it falls off the cliff
- setting: $\alpha = 0.5$, $\disc = 1$, $500$ episodes, $100$ runs, fixed $\epsilon = 0.1$


Why the paths differ
- SARSA's target uses the next action the behavior policy actually takes
    - near the cliff an exploratory step can fall off, and the large negative reward lowers the values that lead there
    - the lowered values spread to the states along the edge, so SARSA's greedy path moves away from the cliff
- Q-learning's target uses the max over the next actions
    - it ignores the exploratory step in the target, so the edge keeps its high value
    - its greedy path is the best one for an agent that stops exploring, but the exploring agent falls more often

Neither is wrong: they answer different questions
- Q-learning: what is the best path if I stop exploring?
- SARSA: what is the best path given that I keep exploring?


**Aside (for your own review): convergence conditions, cliff details and an animation**

- in the tabular case, Q-learning converges to $\actvalfun_*$ for any fixed $\epsilon > 0$, if every state-action pair keeps being visited and the step size decays suitably
- SARSA converges to $\actvalfun_*$ only if $\epsilon \to 0$ as well
- the cliff statistics: Q-learning's greedy path is the 13-step edge route in all 100 runs; SARSA's has 17 steps in 77 of 100 runs (the others learn a 19-step path or a route that is not finished)
- animation of the greedy paths after training: Q-learning (orange) hugs the cliff, SARSA (blue) goes around; settings $\alpha = 0.2$, $\disc = 0.99$, $\epsilon$ decaying to $0.01$, $5000$ episodes; it loops

<img src="images/cliffwalking_paths_qlearning.gif" width=80%>


## SARSA vs Q-learning: summary

| Feature | SARSA (on-policy) | Q-learning (off-policy) |
|:--|:--|:--|
| Policy learned about | the $\epsilon$-greedy behavior policy | the greedy policy |
| Update target | $\rewseq + \disc\,\actvalfun(\state', \act')$ (the action actually taken next) | $\rewseq + \disc \max{\act'} \actvalfun(\state', \act')$ (the best next action) |
| Exploration risk | included in the values (safer paths) | ignored in the values (shortest path, riskier while exploring) |
| Can reuse stored transitions from an older policy (*replay*)? | not naturally: the target needs the action the *current* policy takes | yes: the one-step target does not use the next action actually taken; the next step stores transitions and reuses them |


# Learning via code (for your own review)

[Here](RL_OnPolicy_vs_OffPolicy_code_examples.ipynb) is a notebook that demonstrates
- Q-learning, with and without replay memory
- SARSA

What each example shows
- **Q-learning (off-policy):** the behavior policy ($\epsilon$-greedy) differs from the target policy (greedy); the update uses the greedy maximum even when the sampled action was exploratory
- **SARSA (on-policy):** both policies are $\epsilon$-greedy and match; the update uses the exact next action chosen
- **Q-learning with an experience replay buffer:** stored transitions are reused for updates (the next step)


# What we established, and what it sets up

- **Established**
    - with a model, state values give a policy by lookahead; without one, we learn *action values* and read the policy off them
    - an agent must try actions: $\epsilon$-greedy explores with probability $\epsilon$
    - Q-learning is the sampled value iteration for $\actvalfun$: it learns the greedy policy while acting $\epsilon$-greedily
    - every update has a **behavior policy** (generates the data) and a **target policy** (is learned about): equal means **on-policy** (SARSA), different means **off-policy** (Q-learning)
    - the two learn different things; on a cliff, different paths
- **Sets up**
    - next step starts from the limitation of a table: Microduck's state is $61$ continuous numbers, far too many to visit or store. A neural network learns $\actvalfun$ (deep Q-learning; the same move could be applied to the policy itself), and *replay* (storing past transitions and reusing them for updates) is legal *because* Q-learning is off-policy
    - after that: the argmax over actions does not handle continuous or stochastic actions, so we learn the policy directly (policy gradient); bias and variance of the targets return there
